# Stage Reach Probability Sandbox

This notebook estimates the probability that each nation reaches each tournament stage.

Method:
- simulate every group from the current fixtures and current model
- rebuild the official round-of-32 bracket for that simulated table
- let the best-eight-third-place routing create the round-of-32 uncertainty
- simulate the knockout bracket forward once each iteration's path is known

Any matches already locked in the adaptive state store are respected.

In [1]:
from __future__ import annotations

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import plotly.express as px

from src.adaptive.state_machine import MatchStateMachine
from src.data.loaders import load_fixtures
from src.models.train import load_or_train_ensemble
from src.simulation.stage_reach import (
    build_uncertain_round_of_32_paths,
    estimate_stage_reach_probabilities,
    show_round_of_32_opponents,
)

pd.options.display.max_columns = 50
pd.options.display.float_format = lambda value: f"{value:.4f}"

In [2]:
ITERATIONS = 2000
SEED = 42

model = load_or_train_ensemble()
fixtures = load_fixtures()
resolved_results = MatchStateMachine().resolved_results()
teams = sorted(set(fixtures["home_team"]) | set(fixtures["away_team"]))

print(f"Teams: {len(teams)}")
print(f"Group-stage fixtures: {len(fixtures)}")
print(f"Resolved matches already locked: {len(resolved_results)}")

Teams: 48
Group-stage fixtures: 72
Resolved matches already locked: 0


In [3]:
result = estimate_stage_reach_probabilities(
    fixtures=fixtures,
    model=model,
    iterations=ITERATIONS,
    seed=SEED,
    resolved_results=resolved_results,
)

stage_probabilities = result.stage_probabilities
round_of_32_opponents = result.round_of_32_opponents
best_third_group_mix = result.best_third_group_mix

stage_probabilities.head(20)

,team,finish_1,finish_2,finish_3,best_third,reach_round_of_32,reach_round_of_16,reach_quarter_finals,reach_semi_finals,reach_third_place,reach_final,champion
0,Morocco,0.4640,0.2845,0.1895,0.1540,0.9025,0.6270,0.4430,0.3075,0.1160,0.1915,0.1270
1,Argentina,0.4190,0.3180,0.1780,0.1335,0.8705,0.6200,0.4265,0.2835,0.0975,0.1860,0.1245
2,Portugal,0.5360,0.2270,0.1495,0.1105,0.8735,0.6400,0.4295,0.2560,0.1025,0.1535,0.0995
3,England,0.7510,0.1895,0.0440,0.0345,0.9750,0.5830,0.3720,0.2245,0.0995,0.1250,0.0595
4,Iran,0.3750,0.2970,0.2135,0.1625,0.8345,0.5775,0.3445,0.1935,0.0825,0.1110,0.0585
5,Belgium,0.3255,0.2940,0.2305,0.1730,0.7925,0.5355,0.3215,0.1740,0.0795,0.0945,0.0515
6,Algeria,0.3730,0.3225,0.1935,0.1415,0.8370,0.5370,0.3260,0.1745,0.0695,0.1050,0.0500
7,Japan,0.5060,0.2515,0.1550,0.1205,0.8780,0.4420,0.2750,0.1520,0.0755,0.0765,0.0385
8,Ivory Coast,0.4285,0.2700,0.1820,0.1400,0.8385,0.4720,0.2450,0.1350,0.0640,0.0710,0.0385
9,Spain,0.6370,0.2125,0.1045,0.0820,0.9315,0.4710,0.2845,0.1575,0.0740,0.0835,0.0380


In [4]:
stage_columns = [
    "reach_round_of_32",
    "reach_round_of_16",
    "reach_quarter_finals",
    "reach_semi_finals",
    "reach_third_place",
    "reach_final",
    "champion",
]
stage_labels = {
    "reach_round_of_32": "Round of 32",
    "reach_round_of_16": "Round of 16",
    "reach_quarter_finals": "Quarter-finals",
    "reach_semi_finals": "Semi-finals",
    "reach_third_place": "Third-place match",
    "reach_final": "Final",
    "champion": "Champion",
}

top_n_teams = 24
stage_chart_data = (
    stage_probabilities.head(top_n_teams)
    .set_index("team")[stage_columns]
    .rename(columns=stage_labels)
)

fig = px.imshow(
    stage_chart_data,
    aspect="auto",
    color_continuous_scale="YlGnBu",
    labels={"x": "Stage", "y": "Team", "color": "Probability"},
    title=f"Top {top_n_teams} Teams: Probability to Reach Each Stage",
)
fig.update_layout(height=700)
fig.show()

In [5]:
uncertain_round_of_32_paths = build_uncertain_round_of_32_paths(
    stage_probabilities,
    round_of_32_opponents,
)

uncertain_round_of_32_paths.head(20)

,possible_round_of_32_opponents,reach_round_of_32,reach_round_of_16,champion
France,38,0.7620,0.4480,0.0270
Norway,37,0.7150,0.3890,0.0170
Germany,36,0.6550,0.3295,0.0110
Senegal,35,0.7810,0.4540,0.0325
Ecuador,35,0.5780,0.2480,0.0055
Iraq,35,0.4305,0.1605,0.0030
Ivory Coast,33,0.8385,0.4720,0.0385
Curaçao,33,0.6320,0.2685,0.0040
Canada,28,0.9175,0.4985,0.0295
United States,28,0.8360,0.4175,0.0115


In [6]:
best_third_group_mix

,group,share_of_best_third_slots
0,C,0.0891
1,E,0.0879
2,G,0.0865
3,I,0.0861
4,A,0.0853
5,B,0.0851
6,J,0.0848
7,F,0.0839
8,D,0.0819
9,K,0.0806


In [7]:
heatmap_teams = stage_probabilities.query("reach_round_of_32 > 0").head(16)["team"].tolist()
round_of_32_heatmap = round_of_32_opponents.loc[heatmap_teams, heatmap_teams]

fig = px.imshow(
    round_of_32_heatmap,
    aspect="auto",
    color_continuous_scale="Sunsetdark",
    labels={"x": "Possible opponent", "y": "Team", "color": "Probability"},
    title="Round-of-32 Opponent Heatmap for Top 16 Qualification Teams",
)
fig.update_layout(height=700)
fig.show()

In [8]:
show_round_of_32_opponents(round_of_32_opponents, "Argentina")

Spain                    0.2945
Uruguay                  0.2160
Saudi Arabia             0.1195
Cape Verde               0.1070
Canada                   0.0330
Switzerland              0.0220
Australia                0.0155
Qatar                    0.0115
England                  0.0075
Belgium                  0.0075
United States            0.0070
Iran                     0.0070
Turkey                   0.0055
Egypt                    0.0045
Bosnia and Herzegovina   0.0020
New Zealand              0.0020
Name: Argentina, dtype: float64

## Notes

- `reach_round_of_32` is the qualification probability from the groups.
- The round-of-32 opponent matrix is where the best-third-place chaos shows up.
- Once a specific round-of-32 bracket is formed inside an iteration, the later knockout path is structurally fixed again.